# Build Differential Inputs (Self-Contained)

Run this notebook first to generate pairwise differential input workbooks and manifests.
This notebook is self-contained: core transformation code is defined in notebook cells.

Suggested order:
1. `build_differential_inputs.ipynb` (this notebook)
2. `01_estimate_differential_lrs.ipynb`


In [ ]:
from __future__ import annotations

from collections.abc import Iterable
from dataclasses import dataclass
from datetime import UTC, datetime
import hashlib
import json
from itertools import combinations
from pathlib import Path
import re
from typing import Any

import pandas as pd
import yaml


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root.")

REPO_ROOT = find_repo_root()
CONFIG_PATH = REPO_ROOT / "config/lr_differential_scenarios.yaml"
DRY_RUN = False

REPO_ROOT, CONFIG_PATH


In [ ]:
DEFAULT_KEY_FEATURE_PATTERN = r"^key feature"
PARENTHETICAL_RE = re.compile(r"\(([^)]*)\)")
EG_PREFIX_RE = re.compile(r"^\s*(e\.?\s*g\.?|eg)\.?\s*:?\s*", re.IGNORECASE)


@dataclass(frozen=True)
class ScenarioConfig:
    scenario_id: str
    source_workbook: str
    sheet_name: str
    parser_profile: str
    key_feature_token: str
    pair_scope: str
    expected_category_count: int | None
    exclude_categories: list[str]
    exemplar_strategy: str
    max_exemplars_per_category: int


@dataclass(frozen=True)
class CategoryBlock:
    label: str
    start_col: int
    end_col: int


@dataclass(frozen=True)
class ParsedMatrix:
    category_row_idx: int
    key_feature_row_idx: int
    findings_start_row_idx: int
    categories: list[str]
    category_blocks: list[CategoryBlock]
    findings: list[str]
    warnings: list[str]


def normalize_cell(value: object) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(1_048_576)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def extract_parenthetical_exemplars(category_label: str) -> list[str]:
    match = PARENTHETICAL_RE.search(category_label)
    if not match:
        return []

    inner = EG_PREFIX_RE.sub("", match.group(1)).strip()
    if not inner:
        return []

    exemplars = [part.strip() for part in re.split(r"[;,]", inner)]
    return [item for item in exemplars if item]


def exemplars_for_category(
    category_label: str,
    *,
    strategy: str,
    max_exemplars: int,
) -> list[str]:
    if max_exemplars < 1:
        raise ValueError("max_exemplars must be >= 1")

    if strategy == "parse_parenthetical":
        values = extract_parenthetical_exemplars(category_label)
    else:
        raise ValueError(f"Unsupported exemplar strategy: {strategy}")

    if not values:
        values = [category_label]

    values = values[:max_exemplars]
    if len(values) < max_exemplars:
        values.extend([""] * (max_exemplars - len(values)))
    return values


def find_key_feature_row(df: pd.DataFrame, key_feature_pattern: str) -> int:
    regex = re.compile(key_feature_pattern, flags=re.IGNORECASE)
    for row_idx, raw in enumerate(df.iloc[:, 0].tolist()):
        if regex.search(normalize_cell(raw)):
            return row_idx
    raise ValueError(
        f"Could not find key-feature row in column 0 using pattern {key_feature_pattern!r}."
    )


def _row_categories(df: pd.DataFrame, row_idx: int) -> list[str]:
    row = df.iloc[row_idx].ffill()
    labels: list[str] = []
    for col_idx in range(1, len(row)):
        label = normalize_cell(row.iloc[col_idx])
        if label and label not in labels:
            labels.append(label)
    return labels


def find_category_row(df: pd.DataFrame, *, key_feature_row_idx: int, parser_profile: str) -> int:
    if parser_profile == "matrix_simple":
        return 0

    if parser_profile == "matrix_with_preamble":
        best_row: int | None = None
        best_score = 0

        for row_idx in range(0, key_feature_row_idx):
            labels = _row_categories(df, row_idx)
            if len(labels) < 2:
                continue
            if all(label.lower().startswith("tier ") for label in labels):
                continue

            score = len(labels)
            if score > best_score:
                best_score = score
                best_row = row_idx

        if best_row is None:
            raise ValueError(
                "Could not infer category row for parser_profile='matrix_with_preamble'."
            )
        return best_row

    raise ValueError(f"Unsupported parser_profile: {parser_profile}")


def extract_category_blocks(df: pd.DataFrame, *, category_row_idx: int) -> list[CategoryBlock]:
    categories_row = df.iloc[category_row_idx].ffill()
    blocks: list[CategoryBlock] = []

    current_label: str | None = None
    start_col: int | None = None
    for col_idx in range(1, len(categories_row)):
        label = normalize_cell(categories_row.iloc[col_idx])
        if not label:
            continue
        if current_label is None:
            current_label = label
            start_col = col_idx
            continue
        if label != current_label:
            assert start_col is not None
            blocks.append(CategoryBlock(label=current_label, start_col=start_col, end_col=col_idx))
            current_label = label
            start_col = col_idx

    if current_label is not None and start_col is not None:
        blocks.append(
            CategoryBlock(
                label=current_label,
                start_col=start_col,
                end_col=len(categories_row),
            )
        )

    if not blocks:
        raise ValueError("No category blocks detected in category row.")
    return blocks


def parse_matrix_sheet(
    df: pd.DataFrame,
    *,
    parser_profile: str,
    key_feature_pattern: str = DEFAULT_KEY_FEATURE_PATTERN,
    expected_category_count: int | None = None,
) -> ParsedMatrix:
    key_feature_row_idx = find_key_feature_row(df, key_feature_pattern)
    category_row_idx = find_category_row(
        df, key_feature_row_idx=key_feature_row_idx, parser_profile=parser_profile
    )
    findings_start_row_idx = key_feature_row_idx + 1

    category_blocks = extract_category_blocks(df, category_row_idx=category_row_idx)
    categories = [block.label for block in category_blocks]

    findings: list[str] = []
    for row_idx in range(findings_start_row_idx, df.shape[0]):
        finding = normalize_cell(df.iat[row_idx, 0])
        if finding:
            findings.append(finding)

    warnings: list[str] = []
    if expected_category_count is not None and len(categories) != expected_category_count:
        warnings.append(
            "Detected category count does not match expected count: "
            f"expected={expected_category_count}, observed={len(categories)}"
        )

    return ParsedMatrix(
        category_row_idx=category_row_idx,
        key_feature_row_idx=key_feature_row_idx,
        findings_start_row_idx=findings_start_row_idx,
        categories=categories,
        category_blocks=category_blocks,
        findings=findings,
        warnings=warnings,
    )


def build_pair_sheet(
    *,
    left_category: str,
    right_category: str,
    left_exemplars: list[str],
    right_exemplars: list[str],
    findings: list[str],
    width_per_category: int,
) -> pd.DataFrame:
    if width_per_category < 1:
        raise ValueError("width_per_category must be >= 1")

    total_cols = width_per_category * 2
    total_rows = len(findings) + 2
    rows: list[list[object]] = [[None] * total_cols for _ in range(total_rows)]

    rows[0][0] = left_category
    rows[0][width_per_category] = right_category

    for idx, item in enumerate(left_exemplars[:width_per_category]):
        rows[1][idx] = item
    for idx, item in enumerate(right_exemplars[:width_per_category]):
        rows[1][width_per_category + idx] = item

    for idx, finding in enumerate(findings, start=2):
        rows[idx][0] = finding

    return pd.DataFrame(rows)


def iter_category_pairs(categories: Iterable[str], *, pair_scope: str) -> list[tuple[str, str]]:
    unique = list(dict.fromkeys(categories))
    if pair_scope != "all":
        raise ValueError(f"Unsupported pair_scope: {pair_scope}")
    return list(combinations(unique, 2))


def slugify(text: str, max_len: int = 48) -> str:
    slug = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_").lower()
    slug = re.sub(r"_+", "_", slug)
    slug = slug[:max_len].strip("_")
    return slug or "category"


def pair_filename(pair_index: int, left_category: str, right_category: str) -> str:
    left_slug = slugify(left_category)
    right_slug = slugify(right_category)
    return f"{pair_index:03d}__{left_slug}__vs__{right_slug}.xlsx"


In [ ]:
def load_config(config_path: Path) -> list[ScenarioConfig]:
    raw = yaml.safe_load(config_path.read_text(encoding="utf-8")) or {}
    scenarios = raw.get("scenarios")
    if not isinstance(scenarios, list) or not scenarios:
        raise ValueError("Config must define a non-empty `scenarios` list.")

    configs: list[ScenarioConfig] = []
    for item in scenarios:
        if not isinstance(item, dict):
            raise ValueError("Each scenario entry must be a mapping.")
        configs.append(
            ScenarioConfig(
                scenario_id=str(item["scenario_id"]),
                source_workbook=str(item["source_workbook"]),
                sheet_name=str(item["sheet_name"]),
                parser_profile=str(item.get("parser_profile", "matrix_simple")),
                key_feature_token=str(item.get("key_feature_token", r"^key feature")),
                pair_scope=str(item.get("pair_scope", "all")),
                expected_category_count=(
                    int(item["expected_category_count"])
                    if item.get("expected_category_count") is not None
                    else None
                ),
                exclude_categories=[str(x) for x in item.get("exclude_categories", [])],
                exemplar_strategy=str(item.get("exemplar_strategy", "parse_parenthetical")),
                max_exemplars_per_category=int(item.get("max_exemplars_per_category", 4)),
            )
        )
    return configs


def as_repo_relative(path: Path, repo_root: Path) -> str:
    try:
        return path.resolve().relative_to(repo_root).as_posix()
    except ValueError:
        return path.resolve().as_posix()


def build_outputs(
    *,
    repo_root: Path,
    config_path: Path,
    dry_run: bool,
) -> tuple[int, list[str], Path, Path]:
    configs = load_config(config_path)

    processed_root = repo_root / "data" / "processed" / "lr_differential"
    manifests_dir = processed_root / "manifests"
    inputs_root = processed_root / "inputs"
    outputs_root = processed_root / "outputs"
    manifests_dir.mkdir(parents=True, exist_ok=True)
    inputs_root.mkdir(parents=True, exist_ok=True)
    outputs_root.mkdir(parents=True, exist_ok=True)

    all_rows: list[dict[str, Any]] = []
    all_warnings: list[str] = []
    scenario_manifest: list[dict[str, Any]] = []

    for scenario in configs:
        source_path = (repo_root / scenario.source_workbook).resolve()
        if not source_path.exists():
            raise FileNotFoundError(
                f"Scenario {scenario.scenario_id}: source workbook missing: {source_path}"
            )

        sheet_df = pd.read_excel(source_path, sheet_name=scenario.sheet_name, header=None)
        parsed = parse_matrix_sheet(
            sheet_df,
            parser_profile=scenario.parser_profile,
            key_feature_pattern=scenario.key_feature_token,
            expected_category_count=scenario.expected_category_count,
        )

        categories = [c for c in parsed.categories if c not in set(scenario.exclude_categories)]
        missing_exclusions = sorted(set(scenario.exclude_categories) - set(parsed.categories))
        scenario_warnings = [*parsed.warnings]
        for missing in missing_exclusions:
            scenario_warnings.append(
                "Scenario "
                f"{scenario.scenario_id}: exclude_categories contains unknown label: {missing}"
            )

        pairs = iter_category_pairs(categories, pair_scope=scenario.pair_scope)
        scenario_input_dir = inputs_root / scenario.scenario_id
        scenario_output_dir = outputs_root / scenario.scenario_id
        scenario_input_dir.mkdir(parents=True, exist_ok=True)
        scenario_output_dir.mkdir(parents=True, exist_ok=True)

        for pair_idx, (left, right) in enumerate(pairs, start=1):
            filename = pair_filename(pair_idx, left, right)
            input_path = scenario_input_dir / filename
            output_path = scenario_output_dir / filename.replace(".xlsx", "_filled.xlsx")

            left_examples = exemplars_for_category(
                left,
                strategy=scenario.exemplar_strategy,
                max_exemplars=scenario.max_exemplars_per_category,
            )
            right_examples = exemplars_for_category(
                right,
                strategy=scenario.exemplar_strategy,
                max_exemplars=scenario.max_exemplars_per_category,
            )

            pair_df = build_pair_sheet(
                left_category=left,
                right_category=right,
                left_exemplars=left_examples,
                right_exemplars=right_examples,
                findings=parsed.findings,
                width_per_category=scenario.max_exemplars_per_category,
            )
            if not dry_run:
                pair_df.to_excel(input_path, index=False, header=False)

            all_rows.append(
                {
                    "scenario_id": scenario.scenario_id,
                    "pair_index": pair_idx,
                    "source_workbook": as_repo_relative(source_path, repo_root),
                    "source_sheet": scenario.sheet_name,
                    "parser_profile": scenario.parser_profile,
                    "category_left": left,
                    "category_right": right,
                    "findings_count": len(parsed.findings),
                    "input_workbook": as_repo_relative(input_path, repo_root),
                    "output_workbook": as_repo_relative(output_path, repo_root),
                }
            )

        for warning in scenario_warnings:
            message = f"WARNING: {scenario.scenario_id}: {warning}"
            all_warnings.append(message)
            print(message)

        scenario_manifest.append(
            {
                "scenario_id": scenario.scenario_id,
                "source_workbook": as_repo_relative(source_path, repo_root),
                "source_sheet": scenario.sheet_name,
                "source_sha256": sha256_file(source_path),
                "category_row_idx": parsed.category_row_idx,
                "key_feature_row_idx": parsed.key_feature_row_idx,
                "findings_count": len(parsed.findings),
                "observed_category_count": len(parsed.categories),
                "expected_category_count": scenario.expected_category_count,
                "pair_count": len(pairs),
                "warnings": scenario_warnings,
            }
        )

    pairs_manifest_path = manifests_dir / "pairs_manifest.csv"
    pairs_df = pd.DataFrame(all_rows)
    if not pairs_df.empty:
        pairs_df = pairs_df.sort_values(["scenario_id", "pair_index"]).reset_index(drop=True)
    if not dry_run:
        pairs_df.to_csv(pairs_manifest_path, index=False)

    pairs_manifest_hash = (
        sha256_file(pairs_manifest_path) if not dry_run and pairs_manifest_path.exists() else None
    )

    run_manifest = {
        "generated_at_utc": datetime.now(UTC).isoformat(),
        "config_path": as_repo_relative(config_path.resolve(), repo_root),
        "config_sha256": sha256_file(config_path),
        "pairs_manifest_path": as_repo_relative(pairs_manifest_path, repo_root),
        "pairs_manifest_sha256": pairs_manifest_hash,
        "scenario_count": len(configs),
        "pair_count": len(all_rows),
        "dry_run": dry_run,
        "scenarios": scenario_manifest,
        "warnings": all_warnings,
    }

    run_manifest_path = manifests_dir / "run_manifest.json"
    if not dry_run:
        run_manifest_path.write_text(
            json.dumps(run_manifest, indent=2, sort_keys=True),
            encoding="utf-8",
        )

    print(f"Scenarios processed: {len(configs)}")
    print(f"Pairs generated: {len(all_rows)}")
    print(f"Pairs manifest: {pairs_manifest_path}")
    print(f"Run manifest: {run_manifest_path}")

    return len(all_rows), all_warnings, pairs_manifest_path, run_manifest_path


In [ ]:
pair_count, warnings, pairs_manifest_path, run_manifest_path = build_outputs(
    repo_root=REPO_ROOT,
    config_path=CONFIG_PATH,
    dry_run=DRY_RUN,
)

print({
    "pair_count": pair_count,
    "warning_count": len(warnings),
    "pairs_manifest_path": str(pairs_manifest_path),
    "run_manifest_path": str(run_manifest_path),
})

if pairs_manifest_path.exists():
    display(pd.read_csv(pairs_manifest_path).head(5))
